# Functions in Python

## What are functions, and why are they useful?

So far, you might have used some of the built-in Python functions (e.g. `print()`, `sorted()`), and functions like `math.sin()` or `math.sqrt()` which come with the `math` module. In fact, these functions are _short-hands_ or _interfaces_ for bigger pieces of code which are hidden away. The key idea is _modularity_: you can re-use that functionality easily any time you need to, and you can reproduce the same set of computations with different input values, without having to re-write all the hidden code every single time.

When you _call_ a function, e.g. `math.sqrt(25)`, Python finds the corresponding code in the `math` module for the `sqrt()` function, and executes it, in this case to calculate the square root of `25` and give you the result, `5`. One nice feature is that you can plug in a different number, and it will execute the same code to give you the result for that number as well; thankfully you don't have to go dig into that hidden code and change the number every time you want to calculate a different square root.

Now, as you start writing longer and more complex programs, this is going to be a very useful idea. In this notebook, you will learn to create **your own custom functions** to structure your programs, and avoid having to copy blocks of code again and again if you want to re-use the same computations in multiple different situations.

> _Note: If you know about functions in mathematics, this is actually a very similar concept. A function is an object that takes some input value(s), and transforms the input in a specific, repeatable way to give the corresponding output._

In [ ]:
# Run this cell to import the necessary tools
from wav_library import *
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, Audio

import matplotlib.pyplot as plt

## Playing with sound

It's useful to have a concrete example to understand how this all works; for the rest of this activity, let's put on our headphones, and play with some audio files.

Some short music samples are provided in `guitar.wav`, `piano.wav`, and `synth.wav`. These are audio files in the WAVE format (with the `.wav` extension), which you can open and play in your favourite media player. We can also use Python to open the files and play the sound, thanks to some provided libraries which we don't need to worry about just now -- run the code cell below to play `guitar.wav`:

In [ ]:
guitar = read_wav_file('guitar.wav')
Audio(data=guitar, rate=44100, normalize=False)

The sound you hear is the data contained in the list `guitar`, which is a sequence of more than 600,000 numbers oscillating around 0.

In [ ]:
print(len(guitar))
print(guitar[:10])

If we plot this, we can see that these numbers make up a waveform:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(guitar)
plt.show()

Note that the `rate` corresponds to what we call the _audio sample rate_; it tells us how many samples (numbers) make up 1 second of audio, and it's expressed in Hz. It's an inherent property of audio files; 44100Hz is a common standard, and it happens to be the sample rate of all 3 of the provided samples. We can see that the audio file is around 14 seconds long, which corresponds to `len(guitar) / 44100`.

The actual numbers in `guitar` correspond to the _amplitude_ of the audio signal, i.e. how loud it is. We can make the sound quieter or louder by decreasing or increasing the amplitude (as long as we don't make any number greater than 1 in absolute value -- another important property of audio files is that the maximum possible amplitude is 1). This still needs to be centred around 0, so we use multiplication to change the amplitude by the same amount for the positive and the negative values.

In [ ]:
# Make the sound 10x quieter
guitar_quiet = [sample / 10 for sample in guitar]
Audio(data=guitar_quiet, rate=44100, normalize=False)

In [ ]:
# Make the sound 1.5x louder
guitar_loud = [1.5 * sample for sample in guitar]
Audio(data=guitar_loud, rate=44100, normalize=False)

Note that the option `normalize=False` is here to make sure that the audio player doesn't automatically always play the sound at a fixed volume, but plays at the actual amplitude given by the numbers.

---

***🚩 Exercise:*** Change the amplitude of the guitar sound so that _only the first 2 seconds_ are 5x quieter, then play the whole sound.

In [ ]:
# SOLUTION

# Calculate how many samples is 2 seconds
n_samples = 2 * 44100

# Make the first n_samples 5x quieter
guitar_first2sec = [sample / 5 for sample in guitar[:n_samples]]
guitar_all = guitar_first2sec + guitar[n_samples:]
Audio(data=guitar_all, rate=44100, normalize=False)

---

## Fading out

In general, we want to avoid brutal transitions like this between different volumes. For example, you are probably familiar with the "fade out" effect, where the volume is progressively lowered over a few seconds at the end of a track.

Let's implement this ourselves, and fade out the guitar sample over its entire duration: all we need to do is decrease the volume, starting from the original volume (i.e. the amplitude multiplied by 1) and ending at zero volume (i.e. the amplitude multiplied by zero), with a smooth transition from the start to the end. To do this, we'll multiply the first sample by 1, then the second sample by 0.99..., etc., until the very last sample which we multiply by 0.

A simple way to determine the exact multiplier for each sample, is to create a list of the same length as the audio file, with numbers decreasing from 1 to 0 in a straight line. Let's call this our _envelope_ (it will _envelop_ the loudness of our entire audio signal).

In [ ]:
n_samples = len(guitar)

# Make sure you understand why "n_samples-1" is necessary here
# to ensure that the last value is indeed 0
envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
print(envelope[:5])
print(envelope[-5:])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(envelope)
plt.show()

Now, let's multiply each sample of `guitar` with the corresponding multiplier of `envelope`:

In [ ]:
guitar_fadeout = [guitar[i] * envelope[i] for i in range(n_samples)]
Audio(data=guitar_fadeout, rate=44100, normalize=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(guitar_fadeout)
plt.show()

---

***🚩 Exercise:*** Change the code above so that the fadeout only happens over the last 4 seconds of the guitar sound, instead of over its entire duration.

In [ ]:
# SOLUTION

# Create the envelope for the fade out
n_samples = 4 * 44100
envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]

# Apply the fadeout only to the last 4 seconds of guitar
start_index = len(guitar) - n_samples
guitar_fadeout = guitar[:start_index] + [guitar[start_index + i] * envelope[i] for i in range(n_samples)]

# Plot the results
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(guitar_fadeout)
plt.show()

Audio(data=guitar_fadeout, rate=44100, normalize=False)

---

***🚩 Exercise:*** Change the code again so that the fadeout happens over 10 seconds, but this time on the `piano.wav` sound.

In [ ]:
# SOLUTION

# Read the wav file for the piano sound
piano = read_wav_file('piano.wav')

# Create the envelope for the fade out
n_samples = 10 * 44100
envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]

# Apply the fadeout only to the last 4 seconds of guitar
start_index = len(piano) - n_samples
piano_fadeout = piano[:start_index] + [piano[start_index + i] * envelope[i] for i in range(n_samples)]

# Plot the results
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(piano_fadeout)
plt.show()

Audio(data=piano_fadeout, rate=44100, normalize=False)

---

## Designing a fade out function

By now, we've started repeating some similar code a few times to do slightly different versions of the same thing. The only things we really want to be able to change easily is the audio file, and the duration over which the fade out should happen. The core computation of creating an envelope and multiplying each guitar sample by the corresponding number in the envelope is more or less always the same; it would be great to write it once and be able to reuse it without copying the code over.

This is what **functions** can help with. To design a function, we need to make some key decisions around what we want to be able to change every time, what operations it will perform that can always stay the same, and finally what the function should give us back. There is usually not just one "good" way to do this, and it's very likely that you will change your mind a few times about how to structure your function, what goes in and what comes out -- usually this is after having used it a few times and realised you could make it more practical.

The general syntax of defining a function in Python is:

```python
def my_func(inputs):
    [function body]
    return outputs
```

where

- `my_func` is the name you choose for your function,
- `inputs` are the (zero or more) input arguments,
- `[function body]` are the commands to execute when the function is _called_ (used),
- `outputs` are the (zero or more) return values or output values,
- the function definition is everything that is indented 1 level (or more) below the `def` line.


### Input arguments: what we want to change

Let's start by deciding **what we would like to be able to change** when doing a fade out. As a starting point, we could decide that we want to change:

- the sound on which it's applied,
- the duration of the fade out.

So, if we wanted a 4-second fade out on the guitar sound, we want to make ourselves a function that we could call like this:

```python
fade_out(guitar, 4)
```

But if we had other sounds, we could also use the same function easily:

```python
fade_out(piano, 2.5)
fade_out(synth, 3)
```

We decide that we want 2 **input arguments**, and we give them generic names that will be replaced by the actual objects (e.g. the list `guitar` and the number `4`). So our function definition will start with

```python
def fade_out(audio, duration):
    
```

### Function body: what computations are always done the same way

The **function body** will just be the computations that always happen when we use our function, and that we don't want to have to worry about ever again. For the fade out, looking back at our previous code, we'll need to convert the duration in seconds to a number of samples, then create the envelope, then apply it to the correct part of the audio. We can write these in the function body, knowing that our inputs `audio` and `duration` will be replaced by whatever we decide to provide when we use the function later.

```python
def fade_out(audio, duration):
    # Create the envelope for the fade out (duration is provided as an input)
    n_samples = duration * 44100
    envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
    
    # Apply the fadeout only to the last "duration" seconds of "audio"
    start_index = len(audio) - n_samples
    audio_fadeout = audio[:start_index] + [audio[start_index + i] * envelope[i] for i in range(n_samples)]
```

### Return values: what our function gives us back

Finally, we can decide what our function gives back: this is the **return value(s)**. All the computations done inside the function will be "locked" inside the function, and all of the intermediate values that are produced inside the function (e.g. `n_samples` or `envelope`) will always be **deleted** once the function is done running its instructions, **unless** you explicitly tell your function to give them back to you, by using the `return` keyword. When a function gets to the `return` keyword, it stops immediately, so that always needs to be written at the end.

The decision is then: what do we want back? Here, we probably won't need `n_samples`, `envelope`, or `start_index`; we're really only interested in the transformed audio, which is stored in the variable `audio_fadeout`. So we tell our function to return that:

```python
def fade_out(audio, duration):
    # Create the envelope for the fade out (duration is provided as an input)
    n_samples = duration * 44100
    envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
    
    # Apply the fadeout only to the last "duration" seconds of "audio"
    start_index = len(audio) - n_samples
    audio_fadeout = audio[:start_index] + [audio[start_index + i] * envelope[i] for i in range(n_samples)]

    return audio_fadeout
```

And this is it for our first draft! Let's run the code:

In [ ]:
def fade_out(audio, duration):
    # Create the envelope for the fade out (duration is provided as an input)
    n_samples = duration * 44100
    envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
    
    # Apply the fadeout only to the last "duration" seconds of "audio"
    start_index = len(audio) - n_samples
    audio_fadeout = audio[:start_index] + [audio[start_index + i] * envelope[i] for i in range(n_samples)]

    return audio_fadeout

It looks like nothing happens when you run this code; this is because we've not used our function yet, we've only _defined_ it. We've written the fade out recipe, and Python knows it now so we'll be able to use it. Let's try that:

In [ ]:
guitar_fadeout = fade_out(guitar, 3)
Audio(data=guitar_fadeout, rate=44100, normalize=False)

Success! `guitar_fadeout` is the name we've chosen to give to the list returned by our function, which it knows under the generic name `audio_fadeout`. Let's try a few more:

In [ ]:
# We just pick the first 10 seconds so it's not too long
synth = read_wav_file('synth.wav')[:10*44100]
synth_fadeout = fade_out(synth, 5)
Audio(data=synth_fadeout, rate=44100, normalize=False)

In [ ]:
# Another example...
piano = read_wav_file('piano.wav')
piano_fadeout = fade_out(piano, 3.5)
Audio(data=piano_fadeout, rate=44100, normalize=False)

Oh no, an error! Let's look at the error trace to see what the problem might be. It's in two blocks:

- The first block tells us that the error happened on the line where we used our function, `fade_out(piano, 3.5)`.
- The second block tells us where the problem was inside the function: `range(n_samples)` is highlighted.
- The final line says that this is a `TypeError`, where `range(n_samples)` failed because `n_samples` is not an integer.

Going back the line before, we calculated `n_samples = duration * 44100`. In this example however, we provided `duration` as `3.5`, so `3.5 * 44100` doesn't give an integer.

Since `n_samples` does need to be an integer number of samples, let's add a "safety feature" to our function definition and run it again, in case the duration provided doesn't give an integer. This generalises our code a little bit, and helps our function be more robust instead of crashing out with an error if `n_samples` wasn't an integer. Again, we don't want to have to look at what's inside the function ever again once it's finished and working, so it's good to take these precautions now. This is why it's important to **test** your functions thoroughly.

In [ ]:
def fade_out(audio, duration):
    # Create the envelope for the fade out (duration is provided as an input)
    n_samples = int(duration * 44100)     # make sure it's an integer!
    envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
    
    # Apply the fadeout only to the last "duration" seconds of "audio"
    start_index = len(audio) - n_samples
    audio_fadeout = audio[:start_index] + [audio[start_index + i] * envelope[i] for i in range(n_samples)]

    return audio_fadeout

# We return to the previous indentation level to indicate
# that our function definition is finished
piano_fadeout = fade_out(piano, 3.5)
Audio(data=piano_fadeout, rate=44100, normalize=False)

---

***🚩 Exercise:*** Write another function called `fade_in()` to create a _fade in_ effect, where the amplitude increases progressively from 0 at the start of the sound. Then, use your function to create a 4.5-second fade in on the synth sound.

In [ ]:
# SOLUTION

def fade_in(audio, duration):
    # Create the envelope for the fade in (duration is provided as an input)
    n_samples = int(duration * 44100)     # make sure it's an integer!
    envelope = [i / (n_samples-1) for i in range(n_samples)]
    
    # Apply the fade in only to the first "duration" seconds of "audio"
    audio_fadein = [audio[i] * envelope[i] for i in range(n_samples)] + audio[n_samples:]

    return audio_fadein

synth_fadein = fade_in(synth, 4.5)
Audio(data=synth_fadein, rate=44100, normalize=False)

---

## Default values for input arguments

Python also allows us to give **default values** to some of our input arguments. For example, if we anticipate that most of our fade outs will be 3 seconds long, then we can set this up in the function definition as a default value for `duration`, like this:

```python
def fade_out(audio, duration=3):
    
```

This means that we can now call `fade_out()` using the shorter command `fade_out(audio)` whenever we want to use the default 3-second value. We can also still specify a different value from the default, by calling the function as before, e.g. `fade_out(guitar, 2.5)`. This is useful to make your code more concise and avoid re-typing the same value many times if you use a specific value much more often than others.

In [ ]:
def fade_out(audio, duration=3):
    # Create the envelope for the fade out (duration is provided as an input)
    n_samples = int(duration * 44100)     # make sure it's an integer!
    envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
    
    # Apply the fadeout only to the last "duration" seconds of "audio"
    start_index = len(audio) - n_samples
    audio_fadeout = audio[:start_index] + [audio[start_index + i] * envelope[i] for i in range(n_samples)]

    return audio_fadeout

# Using the default value of 3 seconds
piano_fadeout = fade_out(piano)
Audio(data=piano_fadeout, rate=44100, normalize=False)

In [ ]:
# Using a non-default value still works as before!
synth_fadeout = fade_out(synth, 4.2)
Audio(data=synth_fadeout, rate=44100, normalize=False)

---

***🚩 Exercise:*** Sometimes, audio files don't have the typical 44100Hz sample rate. How would you modify your `fade_` functions to be able to indicate the sample rate if it's different, but still use them the same way if the sample rate is 44100Hz?

In [ ]:
# SOLUTION
# We just add a sample_rate argument with a default value of 44100.

def fade_out(audio, duration=3, sample_rate=44100):
    # Create the envelope for the fade out (duration is provided as an input)
    n_samples = int(duration * sample_rate)     # make sure it's an integer!
    envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
    
    # Apply the fadeout only to the last "duration" seconds of "audio"
    start_index = len(audio) - n_samples
    audio_fadeout = audio[:start_index] + [audio[start_index + i] * envelope[i] for i in range(n_samples)]

    return audio_fadeout

---

***🚩 Exercise:*** A _cross-fade_ is a technique to blend one piece of music gradually into another. It is done by fading out the first piece, fading in the second piece over the same duration, and adding the faded sample values in each piece together so that the fade durations overlap. The fade out and fade in envelopes cross over like this:

In [ ]:
n_samples = 1000
envelope_out = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
envelope_in = [i / (n_samples-1) for i in range(n_samples)]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(envelope_out, label='Fade out envelope')
ax.plot(envelope_in, label='Fade in envelope')
ax.legend(); plt.show()

In the code cell below, write a function called `cross_fade()`, which takes 3 input arguments:

- `audio_file_1` and `audio_file_2`, two strings representing the name of two WAVE files in the current folder,
- `duration`, the duration of the cross-fade, with a default value of 4 seconds.

Your function will first need to call `read_wav_file()` to read the two WAVE files into lists of samples, then call your functions `fade_out()` and `fade_in()` to apply the fade out on the first audio and the fade in on the second audio, and finally create a new list of audio samples, starting with the first audio and finishing with the second, where the faded samples are added together so that they overlap.

For example, if the cross-fade duration is 4 seconds, then the 4-second fade out of the first audio should overlap with the 4-second fade in of the second audio. In other words, the first faded-out sample of the second audio should be added to the first faded-in sample of second audio; etc.

_Bonus task:_ before returning the list of samples corresponding to your cross-faded audio file, use the function `write_wav_file()` from `wav_library()` to also write the result to a WAVE file in the current folder, which you can open to play like any other audio file. You will need to look at the function in `wav_library.py` to figure out what are the input arguments you should use. You can add another input argument to your function to indicate what file name you want to use to write the file.

_Hint: you will need to know `n_samples` to be able to add the correct samples together from both files. Calling your `fade_` functions doesn't return that value; you can choose to recalculate it in `cross_fade()`, or to change your `fade_` functions so that they also return `n_samples`. The choice is yours!_

In [ ]:
# Define your function here!




# For testing: this should play the two sounds cross-faded one after the other over 6 seconds,
# and an output file should appear in the same folder
guitar_piano_crossfade = cross_fade('guitar.wav', 'piano.wav', 6)
Audio(data=guitar_piano_crossfade, rate=44100, normalize=False)

In [ ]:
# SOLUTION
def cross_fade(audio_file_1, audio_file_2, duration=4, output_filename='crossfade.wav'):
    # Read the two audio files into lists of samples
    audio_1 = read_wav_file(audio_file_1)
    audio_2 = read_wav_file(audio_file_2)

    # Apply fades to each file
    audio_1_fade = fade_out(audio_1, duration)
    audio_2_fade = fade_in(audio_2, duration)

    # Recalculate n_samples from duration
    n_samples = int(duration * 44100)

    # Create the new cross-faded audio and return it
    cross_faded_part = [s1 + s2 for s1, s2 in zip(audio_1_fade[-n_samples:], audio_2_fade[:n_samples])]
    audio_crossfade = audio_1_fade[:-n_samples] + cross_faded_part + audio_2_fade[n_samples:]

    # Write the output to a WAVE file
    write_wav_file(audio_crossfade, output_filename)

    return audio_crossfade


# For testing: this should play the two sounds cross-faded one after the other over 6 seconds,
# and an output file should appear in the same folder
guitar_piano_crossfade = cross_fade('guitar.wav', 'piano.wav', 6)
Audio(data=guitar_piano_crossfade, rate=44100, normalize=False)

---

***🚩 Exercise:*** We can break down our code into even smaller steps, and write functions for these smaller steps. Write a function `create_envelope()`, which takes one integer argument `n_samples`, and returns a list `envelope` corresponding to the envelope needed for a fade-out over `n_samples` samples.

Then, in your `fade_out()` function, replace the line that creates the envelope by a line calling your new function `create_envelope()`.

How could you use the output of this same function in your `fade_in()` function?

In [ ]:
# SOLUTION

def create_envelope(n_samples):
    envelope = [(n_samples-1 - i) / (n_samples-1) for i in range(n_samples)]
    return envelope


def fade_out(audio, duration=3):
    # Create the envelope for the fade out (duration is provided as an input)
    n_samples = int(duration * 44100)     # make sure it's an integer!
    envelope = create_envelope(n_samples)
    
    # Apply the fadeout only to the last "duration" seconds of "audio"
    start_index = len(audio) - n_samples
    audio_fadeout = audio[:start_index] + [audio[start_index + i] * envelope[i] for i in range(n_samples)]

    return audio_fadeout


def fade_in(audio, duration):
    # Create the envelope for the fade in (duration is provided as an input)
    n_samples = int(duration * 44100)     # make sure it's an integer!
    
    # We need to flip the envelope left-to-right to make it fade-in:
    envelope = create_envelope(n_samples)[::-1]
    
    # Apply the fade in only to the first "duration" seconds of "audio"
    audio_fadein = [audio[i] * envelope[i] for i in range(n_samples)] + audio[n_samples:]

    return audio_fadein

---

_[This is an extension exercise, to practice default function values and calling functions from inside other functions. It needs some knowledge of conditional statements. It could be further extended to include input argument checking, e.g. raise an error if `fade_type` takes an undefined value.]_

***🚩 Exercise:*** Write a function `fade()`, which takes 3 input arguments:

- `audio_file`, a string representing the name of a WAVE file in the current folder,
- `duration`, the duration of the fade, with a default value of 3.5 seconds,
- `fade_type`, a string which can be either `'in'`, `'out'`, or `'both'`, with default value `'out'`.

Your function will first read the WAVE file into a list using the `read_wav_file()` function, then use your `fade_in()` and/or `fade_out()` functions from before to apply a fade in, a fade out, or both, depending on the value of `fade_type`, with duration `duration` seconds, and return the resulting audio after the fade(s) are applied.

In [ ]:
# SOLUTION

def fade(audio_file, duration=3.5, fade_type='out'):
    # Read the audio file
    audio = read_wav_file(audio_file)

    # Apply the fade depending on fade_type
    if fade_type == 'in':
        audio_fade = fade_in(audio, duration)
    elif fade_type == 'out':
        audio_fade = fade_in(audio, duration)
    elif fade_type == 'both':
        audio_fade = fade_in(audio, duration)
        audio_fade = fade_in(audio_fade, duration)

    # Return the output
    return audio_fade